In [ ]:
%matplotlib widget

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider, RadioButtons,
    HTML, HTMLMath,
    VBox, HBox, Layout
)
from IPython.display import display

plt.ioff()

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.container{
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box{
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll{
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure{
    overflow:visible !important;
    resize:none !important;
}

.ip-title{
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.ip-subtitle{
    font-family:Arial, sans-serif;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
}

.ip-label{
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.ip-value{
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

.ip-radio .widget-radio-box{
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    gap:18px !important;
}

.ip-radio > label{
    display:none !important;
}

</style>
"""))

# ============================================================
# GLOBAL WIDTH
# ============================================================

TOTAL_WIDTH = '1120px'

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1120px;
    padding:9px 13px;
    border:1px solid #d2c2df;
    font-family:Arial,sans-serif;
    font-size:15px;
    line-height:1.45;
    box-sizing:border-box;
">

<div class="ip-title" style="margin-bottom:6px;">
Inner Product, Orthogonality and Fourier Coefficients
</div>

<div style="margin-bottom:4px;">
In a function space, orthogonality is defined through the inner product.
For real functions on [−π,π],
</div>

<div style="margin-bottom:4px;text-align:center;">
<b>〈f,g〉 = ∫<sub>−π</sub><sup>π</sup> f(t)g(t)dt.</b>
</div>

<div style="margin-bottom:4px;">
The trigonometric functions form an orthogonal family. Their inner products
play the same role as dot products between ordinary vectors.
</div>

<div>
<b>This notebook:</b> computes inner products symbolically and shows how
a Fourier coefficient is obtained by projection onto a selected basis function.
</div>

</div>
""")

# ============================================================
# SYMBOLS
# ============================================================

t = sp.symbols('t', real=True)

x_symbolic = (
    1
    + sp.Rational(3,2) * sp.cos(2*t)
    - sp.Rational(4,5) * sp.sin(3*t)
)

# ============================================================
# CONTROLS
# ============================================================

pair_selector = RadioButtons(
    options=[
        ('cos–cos', 'cc'),
        ('sin–sin', 'ss'),
        ('cos–sin', 'cs')
    ],
    value='cc',
    description='',
    layout=Layout(width='360px')
)
pair_selector.add_class('ip-radio')

coef_selector = RadioButtons(
    options=[
        ('cos(k t)', 'cos'),
        ('sin(k t)', 'sin')
    ],
    value='cos',
    description='',
    layout=Layout(width='300px')
)
coef_selector.add_class('ip-radio')

n_slider = IntSlider(
    min=1, max=8, step=1, value=2,
    readout=False,
    continuous_update=True,
    layout=Layout(width='230px')
)

m_slider = IntSlider(
    min=1, max=8, step=1, value=3,
    readout=False,
    continuous_update=True,
    layout=Layout(width='230px')
)

k_slider = IntSlider(
    min=1, max=8, step=1, value=2,
    readout=False,
    continuous_update=True,
    layout=Layout(width='230px')
)

n_value = HTML()
m_value = HTML()
k_value = HTML()

def slider_row(label, slider, value):

    return HBox(
        [
            HTML(
                f'<div class="ip-label">{label}</div>',
                layout=Layout(width='35px')
            ),
            slider,
            value
        ],
        layout=Layout(
            width='320px',
            height='32px',
            align_items='center'
        )
    )

# ============================================================
# CONTROL PANELS
# ============================================================

orth_controls = VBox(
    [
        HTML(
            '<div class="ip-subtitle" style="margin-bottom:4px;">'
            'Orthogonality Test'
            '</div>'
        ),
        pair_selector,
        slider_row('n:', n_slider, n_value),
        slider_row('m:', m_slider, m_value)
    ],
    layout=Layout(
        width='552px',
        padding='8px 12px',
        border='1px solid #d2c2df'
    )
)

coef_controls = VBox(
    [
        HTML(
            '<div class="ip-subtitle" style="margin-bottom:4px;">'
            'Fourier Projection'
            '</div>'
        ),
        coef_selector,
        slider_row('k:', k_slider, k_value)
    ],
    layout=Layout(
        width='552px',
        padding='8px 12px',
        border='1px solid #d2c2df'
    )
)

controls_row = HBox(
    [
        orth_controls,
        coef_controls
    ],
    layout=Layout(
        width=TOTAL_WIDTH,
        gap='16px',
        align_items='stretch'
    )
)

# ============================================================
# SYMBOLIC RESULT WIDGETS
# ============================================================

functions_math = HTMLMath()
inner_math = HTMLMath()
norms_math = HTMLMath()
signal_math = HTMLMath()
basis_math = HTMLMath()
coef_math = HTMLMath()

# ============================================================
# SYMBOLIC RESULTS — SINGLE ROW
# ============================================================

symbolic_row = HBox(
    [
        functions_math,
        inner_math,
        norms_math,
        signal_math,
        basis_math,
        coef_math
    ],
    layout=Layout(
        width='1090px',
        justify_content='space-between',
        align_items='center'
    )
)

result_panel = VBox(
    [
        HTML(
            '<div class="ip-subtitle" style="margin-bottom:7px;">'
            'Symbolic Results'
            '</div>'
        ),
        symbolic_row
    ],
    layout=Layout(
        width=TOTAL_WIDTH,
        padding='9px 13px',
        border='1px solid #d2c2df'
    )
)

# ============================================================
# NUMERICAL AXIS
# ============================================================

tt = np.linspace(
    -np.pi,
    np.pi,
    1200
)

# ============================================================
# FIGURE 1
# ============================================================

fig1, ax1 = plt.subplots(
    figsize=(6.2, 4.25)
)

fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False
fig1.canvas.toolbar_visible = False

fig1.canvas.layout = Layout(
    width='620px',
    height='425px',
    margin='0px'
)

ax1.set_title(
    'Selected Basis Functions',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax1.set_xlabel('t')
ax1.set_ylabel('Amplitude')

ax1.set_xlim(
    -np.pi,
    np.pi
)

ax1.set_ylim(
    -1.2,
    1.2
)

ax1.set_xticks(
    [-np.pi, 0, np.pi]
)

ax1.set_xticklabels(
    [r'$-\pi$', '0', r'$\pi$']
)

ax1.axhline(
    0,
    linewidth=0.8
)

ax1.axvline(
    0,
    linewidth=0.8
)

ax1.grid(
    True,
    linestyle=':',
    alpha=0.4
)

f_line, = ax1.plot(
    tt,
    np.cos(2*tt),
    linewidth=2,
    label='f(t)'
)

g_line, = ax1.plot(
    tt,
    np.cos(3*tt),
    linewidth=2,
    label='g(t)'
)

ax1.legend(
    loc='upper right'
)

fig1.subplots_adjust(
    left=0.11,
    right=0.97,
    top=0.88,
    bottom=0.14
)

# ============================================================
# FIGURE 2
# ============================================================

fig2, ax2 = plt.subplots(
    figsize=(4.75, 4.25)
)

fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False
fig2.canvas.toolbar_visible = False

fig2.canvas.layout = Layout(
    width='475px',
    height='425px',
    margin='0px'
)

ax2.set_title(
    'Product f(t)g(t)',
    fontsize=14,
    fontweight='bold',
    color='#0b3d91'
)

ax2.set_xlabel('t')
ax2.set_ylabel('Product')

ax2.set_xlim(
    -np.pi,
    np.pi
)

ax2.set_ylim(
    -1.2,
    1.2
)

ax2.set_xticks(
    [-np.pi, 0, np.pi]
)

ax2.set_xticklabels(
    [r'$-\pi$', '0', r'$\pi$']
)

ax2.axhline(
    0,
    linewidth=0.8
)

ax2.axvline(
    0,
    linewidth=0.8
)

ax2.grid(
    True,
    linestyle=':',
    alpha=0.4
)

product_line, = ax2.plot(
    tt,
    np.cos(2*tt) * np.cos(3*tt),
    linewidth=2
)

fig2.subplots_adjust(
    left=0.14,
    right=0.97,
    top=0.88,
    bottom=0.14
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig1.canvas,
        fig2.canvas
    ],
    layout=Layout(
        width=TOTAL_WIDTH,
        gap='15px',
        align_items='flex-start'
    )
)

# ============================================================
# SELECT FUNCTIONS
# ============================================================

def selected_functions():

    n = n_slider.value
    m = m_slider.value
    mode = pair_selector.value

    if mode == 'cc':

        f = sp.cos(n*t)
        g = sp.cos(m*t)

        fn = np.cos(n*tt)
        gn = np.cos(m*tt)

    elif mode == 'ss':

        f = sp.sin(n*t)
        g = sp.sin(m*t)

        fn = np.sin(n*tt)
        gn = np.sin(m*tt)

    else:

        f = sp.cos(n*t)
        g = sp.sin(m*t)

        fn = np.cos(n*tt)
        gn = np.sin(m*tt)

    return f, g, fn, gn

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    n = n_slider.value
    m = m_slider.value
    k = k_slider.value

    n_value.value = (
        f'<div class="ip-value">{n}</div>'
    )

    m_value.value = (
        f'<div class="ip-value">{m}</div>'
    )

    k_value.value = (
        f'<div class="ip-value">{k}</div>'
    )

    # --------------------------------------------------------
    # Selected functions
    # --------------------------------------------------------

    f, g, fn, gn = selected_functions()

    # --------------------------------------------------------
    # Symbolic inner product
    # --------------------------------------------------------

    inner = sp.simplify(
        sp.integrate(
            sp.expand_trig(f*g),
            (t, -sp.pi, sp.pi)
        )
    )

    norm_f_sq = sp.simplify(
        sp.integrate(
            f**2,
            (t, -sp.pi, sp.pi)
        )
    )

    norm_g_sq = sp.simplify(
        sp.integrate(
            g**2,
            (t, -sp.pi, sp.pi)
        )
    )

    # --------------------------------------------------------
    # Fourier projection
    # --------------------------------------------------------

    if coef_selector.value == 'cos':
        basis = sp.cos(k*t)
    else:
        basis = sp.sin(k*t)

    numerator = sp.simplify(
        sp.integrate(
            sp.expand_trig(
                x_symbolic * basis
            ),
            (t, -sp.pi, sp.pi)
        )
    )

    denominator = sp.simplify(
        sp.integrate(
            basis**2,
            (t, -sp.pi, sp.pi)
        )
    )

    coefficient = sp.simplify(
        numerator / denominator
    )

    # --------------------------------------------------------
    # One-line symbolic results
    # --------------------------------------------------------

    functions_math.value = (
        r'\('
        r'f(t)='
        +
        sp.latex(f)
        +
        r',\;g(t)='
        +
        sp.latex(g)
        +
        r'\)'
    )

    inner_math.value = (
        r'\('
        r'\langle f,g\rangle='
        +
        sp.latex(inner)
        +
        r'\)'
    )

    norms_math.value = (
        r'\('
        r'\|f\|^2='
        +
        sp.latex(norm_f_sq)
        +
        r',\;\|g\|^2='
        +
        sp.latex(norm_g_sq)
        +
        r'\)'
    )

    signal_math.value = (
        r'\('
        r'x(t)='
        +
        sp.latex(x_symbolic)
        +
        r'\)'
    )

    basis_math.value = (
        r'\('
        r'\phi_k(t)='
        +
        sp.latex(basis)
        +
        r'\)'
    )

    coef_latex = sp.latex(
        coefficient
    ).replace(
        r'\frac',
        r'\dfrac'
    )

    coef_math.value = (
        r'\('
        r'c_k='
        r'\dfrac{\langle x,\phi_k\rangle}'
        r'{\langle\phi_k,\phi_k\rangle}'
        r'='
        +
        coef_latex
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------

    f_line.set_ydata(fn)
    g_line.set_ydata(gn)
    product_line.set_ydata(fn*gn)

    fig1.canvas.draw_idle()
    fig2.canvas.draw_idle()

# ============================================================
# CONNECT
# ============================================================

for widget in [
    pair_selector,
    n_slider,
    m_slider,
    coef_selector,
    k_slider
]:
    widget.observe(
        update,
        names='value'
    )

update()

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    width:1120px;
    padding:10px 13px;
    border:1px solid #d7c7e5;
    font-family:Arial,sans-serif;
    font-size:14px;
    line-height:1.52;
    box-sizing:border-box;
    margin-top:5px;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Interpretation
</div>

<div style="margin-bottom:5px;">
The integral of the product is the function-space analogue of the dot product
between ordinary vectors. A zero value therefore means that the selected
functions are orthogonal.
</div>

<div style="margin-bottom:5px;">
The sine and cosine families are mutually orthogonal on [−π,π].
For functions of the same family, the inner product is nonzero only
when the frequency indices coincide.
</div>

<div>
A Fourier coefficient is a projection coordinate: it measures the component
of the signal along the selected basis function.
</div>

</div>
""")

# ============================================================
# FINAL DISPLAY
# ============================================================

display(
    VBox(
        [
            documentation,
            controls_row,
            result_panel,
            figures_row,
            interpretation
        ],
        layout=Layout(
            width='1140px',
            gap='7px',
            align_items='flex-start'
        )
    )
)